#**Proyecto: Big Data**
#Evaluación del módulo Introducción al procesamiento distribuido y sistemas Big Data


###  **Objetivo del Proyecto**

Diseñar e implementar un **pipeline de Big Data con Apache Spark** que:

* Lea datos en formatos JSON, CSV y Parquet.
* Procese datos estructurados y no estructurados.
* Soporte procesamiento batch y en streaming.
* Entrene un modelo de aprendizaje automático escalable con MLlib.

---

## **Lección 1: Big Data**

###  Tareas:

1. **Definir las 5V’s del Big Data aplicadas al caso:**

   * **Volumen**: Datos masivos de ventas, IoT, redes sociales y logs.
   * **Velocidad**: Requiere procesamiento en tiempo real (streaming).
   * **Variedad**: Datos estructurados (ventas) y no estructurados (logs, redes sociales).
   * **Veracidad**: Es necesario limpiar datos ruidosos de sensores/redes.
   * **Valor**: Se busca extraer insights para la toma de decisiones.

2. **Beneficios del enfoque distribuido vs. local:**

   * Escalabilidad horizontal.
   * Tolerancia a fallos.
   * Mayor velocidad al procesar datos en paralelo.

3. **Mapa de tecnologías clave:**

   * Apache Spark (Core, SQL, Streaming, MLlib).
   * HDFS o almacenamiento distribuido (simulado).
   * Kafka o sockets (entrada en tiempo real).
   * JSON, CSV, Parquet (formatos).

---

## **Lección 2: Apache Spark**

###  Tareas:

1. **¿Cuándo usar Spark?**

   * Cuando se necesita procesar datos a gran escala, en memoria y con baja latencia.
   * Cuando se combinan batch, streaming, ML y SQL.

2. **Arquitectura de Spark:**

   * **Driver**: Programa principal, coordina todo.
   * **Executors**: Ejecutan tareas.
   * **Cluster Manager**: Asigna recursos (YARN, Kubernetes, etc.).

3. **Módulos a usar en el proyecto:**

   * Spark Core
   * Spark SQL
   * Spark Streaming
   * MLlib

---

##  **Lección 3: Procesamiento distribuido con RDDs**

###  Tareas:

1. Crear entorno local en Colab o Databricks.
2. Usar `parallelize()` y `textFile()` para cargar datos.
3. Aplicar transformaciones (`map`, `filter`, `sortBy`, etc.).
4. Aplicar acciones (`count`, `collect`, `mean`).
5. Documentar el flujo de ejecución y el Job Spark.

---

##  **Lección 4: Datos estructurados y Spark SQL**

###  Tareas:

1. Cargar datos JSON, CSV, Parquet.
2. Realizar consultas SQL.
3. Usar UDFs (funciones definidas por el usuario).
4. Comparar rendimiento vs RDD.

---

## **Lección 5: Procesamiento en streaming**

###  Tareas:

1. Leer datos en tiempo real desde socket o archivo simulado.
2. Aplicar transformaciones sobre DStreams.
3. Implementar ventana temporal (`window`, `reduceByKeyAndWindow`).
4. Comparar latencia con procesamiento batch.

---

##  **Lección 6: ML Escalable con MLlib**

###  Tareas:

1. Preparar datos: encoding, escalado, selección de features.
2. Entrenar modelo supervisado (ej. clasificación con `LogisticRegression`).
3. Evaluar desempeño (`AUC`, `accuracy`, `RMSE`, etc.).
4. Guardar modelo (`.save()`).
5. Preparar predicción sobre datos batch o streaming.




#**Notebooks/Códigos**

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Colab Spark Default") \
    .getOrCreate()

df = spark.createDataFrame([(1, "ok"), (2, "sin descargas")], ["id", "estado"])
df.show()


+---+-------------+
| id|       estado|
+---+-------------+
|  1|           ok|
|  2|sin descargas|
+---+-------------+



#notebook 01 Ingesta Batch Spark

In [10]:
# -------------------------------------------
# 1. Leer archivo CSV: ventas.csv
ventas = spark.read.option("header", True).option("inferSchema", True).csv("/content/sample_data/ventas.csv")

print("Ventas:")
ventas.show(truncate=False)
ventas.printSchema()

# -------------------------------------------
# 2. Leer archivo JSON: sensores.json
sensores = spark.read.option("multiline", True).json("/content/sample_data/sensores.json")

print("Sensores:")
sensores.show(truncate=False)
sensores.printSchema()

# Convertimos timestamp a formato adecuado
from pyspark.sql.functions import to_timestamp

sensores = sensores.withColumn("timestamp", to_timestamp("timestamp"))

# -------------------------------------------
# 3. Leer archivo de texto: logs.txt (formato no estructurado)
import re

logs_rdd = spark.sparkContext.textFile("/content/sample_data/logs.txt")

# Expresión regular para extraer [timestamp] - [nivel] - [mensaje]
pattern = r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) - (\w+) - (.*)"

logs_df = logs_rdd \
    .map(lambda line: re.findall(pattern, line)) \
    .filter(lambda x: len(x) > 0) \
    .map(lambda x: x[0]) \
    .toDF(["timestamp", "nivel", "mensaje"]) \
    .withColumn("timestamp", to_timestamp("timestamp"))

print("Logs:")
logs_df.show(truncate=False)
logs_df.printSchema()

# -------------------------------------------
# 4. Guardar en disco
ventas.write.mode("overwrite").parquet("/content/sample_data/output/ventas_parquet")
sensores.write.mode("overwrite").json("/content/sample_data/output/sensores_json")
logs_df.write.mode("overwrite").parquet("/content/sample_data/output/logs_parquet")

# -------------------------------------------
# 5. Validar lectura post-exportación
print("✅ Archivos exportados. Verificación:")
spark.read.parquet("/content/sample_data/output/ventas_parquet").show(3)
spark.read.json("/content/sample_data/output/sensores_json").show(3)
spark.read.parquet("/content/sample_data/output/logs_parquet").show(3)


Ventas:
+--------+-------------------+------------------+-----------+-----------+--------+---------------+
|id_venta|fecha              |cliente           |producto   |categoria  |cantidad|precio_unitario|
+--------+-------------------+------------------+-----------+-----------+--------+---------------+
|1       |2025-06-03 04:04:08|Sara Jacobs       |now        |Hogar      |5       |8.77           |
|2       |2025-02-23 05:00:49|Karen Wright      |shake      |Ropa       |5       |189.44         |
|3       |2025-03-12 09:33:42|Elizabeth Warner  |little     |Electrónica|2       |52.83          |
|4       |2025-05-04 07:43:57|Emily Smith       |that       |Hogar      |5       |335.19         |
|5       |2025-01-17 00:05:13|David Wood        |certainly  |Hogar      |1       |160.1          |
|6       |2025-02-20 14:10:00|Mary Turner       |myself     |Libros     |2       |92.01          |
|7       |2025-05-23 17:05:45|Dawn Wilson       |audience   |Ropa       |5       |87.03          |
|8

#Notebook 02: Consultas SQL con Spark SQL

In [12]:
#  Notebook 02: Consultas SQL con Spark

#  Requiere que hayas corrido el Notebook 01 primero.

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, upper, length

# Crear la SparkSession
spark = SparkSession.builder.appName("Consultas SQL con Spark").getOrCreate()

# Cargar los datos desde archivos transformados
ventas_df = spark.read.option("header", True).option("inferSchema", True).csv("sample_data/ventas.csv")
sensores_df = spark.read.option("inferSchema", True).json("sample_data/sensores.json")
logs_rdd = spark.sparkContext.textFile("sample_data/logs.txt")


# Transformar logs RDD a DataFrame
import re
logs_df = logs_rdd.map(lambda line: re.findall(r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) - (\w+) - (.+)", line)) \
    .filter(lambda x: len(x) > 0) \
    .map(lambda x: x[0]) \
    .toDF(["timestamp", "nivel", "mensaje"])

# Cambiar tipo de timestamp (string a timestamp real)
from pyspark.sql.functions import to_timestamp
logs_df = logs_df.withColumn("timestamp", to_timestamp("timestamp"))

# Registrar tablas temporales
ventas_df.createOrReplaceTempView("ventas")
sensores_df.createOrReplaceTempView("sensores")
logs_df.createOrReplaceTempView("logs")

# --------------------------
#  Consultas SQL
# --------------------------

#  1. Total de ventas por categoría
q1 = spark.sql("""
    SELECT categoria, SUM(cantidad * precio_unitario) AS total_ventas
    FROM ventas
    GROUP BY categoria
    ORDER BY total_ventas DESC
""")

#  2. Temperaturas mayores a 30 grados
q2 = spark.sql("""
    SELECT sensor_id, temperatura, humedad, timestamp
    FROM sensores
    WHERE temperatura > 30
    ORDER BY temperatura DESC
""")

#  3. Conteo de logs por nivel de mensaje (INFO, WARN, ERROR)
q3 = spark.sql("""
    SELECT nivel, COUNT(*) AS cantidad
    FROM logs
    GROUP BY nivel
    ORDER BY cantidad DESC
""")

# --------------------------
#  Mostrar resultados
# --------------------------

print("\n Total de ventas por categoría:")
q1.show()

print("\n Temperaturas altas:")
q2.show()

print("\n Logs por nivel:")
q3.show()


 Total de ventas por categoría:
+-----------+------------------+
|  categoria|      total_ventas|
+-----------+------------------+
|     Libros|          19170.75|
|      Hogar|18650.410000000003|
|Electrónica|          15795.36|
|       Ropa|15110.209999999997|
+-----------+------------------+


 Temperaturas altas:
+---------+-----------+-------+-------------------+
|sensor_id|temperatura|humedad|          timestamp|
+---------+-----------+-------+-------------------+
|      S10|       35.0|   64.3|2025-06-03T02:54:55|
|       S2|       34.8|   53.4|2025-04-04T05:04:06|
|       S2|       34.7|   45.4|2025-04-27T07:38:54|
|       S2|       34.2|   49.1|2025-02-16T22:27:25|
|      S10|       34.2|   83.5|2025-07-01T04:49:38|
|       S4|       34.1|   62.6|2025-02-26T11:19:37|
|       S1|       34.0|   46.1|2025-09-10T04:23:17|
|       S4|       34.0|   39.2|2025-01-18T22:38:49|
|       S3|       33.9|   40.7|2025-03-18T18:11:47|
|       S6|       33.7|   58.5|2025-09-11T08:19:21|
|   

#Notebook 03: Streaming con Spark

In [13]:
# Requiere tener una SparkSession ya creada:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, explode, current_timestamp, window
import os

spark = SparkSession.builder.getOrCreate()

# Paso 1: Crear archivo simulado
stream_file_path = "/content/streaming.txt"
with open(stream_file_path, "w") as f:
    f.write("Hola mundo esto es una simulación de streaming\n")
    f.write("Otra línea con más palabras para procesar\n")
    f.write("Y una tercera línea para cerrar\n")

# Paso 2: Leer como archivo de texto (simulación de flujo)
lineas = spark.read.text(stream_file_path)

# Paso 3: Añadir marca de tiempo a cada línea como si fuese en vivo
con_timestamp = lineas.withColumn("timestamp", current_timestamp())

# Paso 4: Separar en palabras
palabras = con_timestamp.select(
    explode(split("value", " ")).alias("palabra"),
    "timestamp"
)

# Paso 5: Aplicar ventana de 10 minutos con desplazamiento de 5 min + watermark
conteo = palabras \
    .withWatermark("timestamp", "5 minutes") \
    .groupBy(window("timestamp", "10 minutes", "5 minutes"), "palabra") \
    .count()

# Paso 6: Mostrar resultado (como si fuese output de writeStream)
print("✅ Conteo por ventana:")
conteo.orderBy("window").show(truncate=False)


✅ Conteo por ventana:
+------------------------------------------+----------+-----+
|window                                    |palabra   |count|
+------------------------------------------+----------+-----+
|{2025-09-17 08:15:00, 2025-09-17 08:25:00}|es        |1    |
|{2025-09-17 08:15:00, 2025-09-17 08:25:00}|streaming |1    |
|{2025-09-17 08:15:00, 2025-09-17 08:25:00}|con       |1    |
|{2025-09-17 08:15:00, 2025-09-17 08:25:00}|una       |2    |
|{2025-09-17 08:15:00, 2025-09-17 08:25:00}|palabras  |1    |
|{2025-09-17 08:15:00, 2025-09-17 08:25:00}|Hola      |1    |
|{2025-09-17 08:15:00, 2025-09-17 08:25:00}|esto      |1    |
|{2025-09-17 08:15:00, 2025-09-17 08:25:00}|tercera   |1    |
|{2025-09-17 08:15:00, 2025-09-17 08:25:00}|de        |1    |
|{2025-09-17 08:15:00, 2025-09-17 08:25:00}|más       |1    |
|{2025-09-17 08:15:00, 2025-09-17 08:25:00}|línea     |2    |
|{2025-09-17 08:15:00, 2025-09-17 08:25:00}|simulación|1    |
|{2025-09-17 08:15:00, 2025-09-17 08:25:00}|proc

#Notebook 4: Machine Learning con Spark MLlib

In [14]:
# ✅ Notebook 04: Machine Learning con Spark MLlib
# Objetivo: Clasificar productos de ventas.csv según su categoría usando MLlib

from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Crear la sesión de Spark
spark = SparkSession.builder.appName("MLlib Ventas").getOrCreate()

# Cargar el dataset de ventas
ventas_df = spark.read.option("header", True).option("inferSchema", True).csv("/content/sample_data/ventas.csv")
ventas_df.show(5)
ventas_df.printSchema()

# Indexar la variable objetivo (categoria)
indexer = StringIndexer(inputCol="categoria", outputCol="categoria_index")
ventas_df = indexer.fit(ventas_df).transform(ventas_df)

# Crear vector de características
features_cols = ["cantidad", "precio_unitario"]
assembler = VectorAssembler(inputCols=features_cols, outputCol="features")
ventas_df = assembler.transform(ventas_df)

# Seleccionar columnas finales para el modelo
final_df = ventas_df.select("features", "categoria_index")

# Dividir en entrenamiento y prueba
train_df, test_df = final_df.randomSplit([0.8, 0.2], seed=42)

# Entrenar modelo Random Forest
rf = RandomForestClassifier(labelCol="categoria_index", featuresCol="features", numTrees=10)
model = rf.fit(train_df)

# Predecir en test
predictions = model.transform(test_df)
predictions.select("features", "categoria_index", "prediction").show(10)

# Evaluar el modelo
evaluator = MulticlassClassificationEvaluator(
    labelCol="categoria_index", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"\n✅ Precisión del modelo: {accuracy:.2%}")

+--------+-------------------+----------------+---------+-----------+--------+---------------+
|id_venta|              fecha|         cliente| producto|  categoria|cantidad|precio_unitario|
+--------+-------------------+----------------+---------+-----------+--------+---------------+
|       1|2025-06-03 04:04:08|     Sara Jacobs|      now|      Hogar|       5|           8.77|
|       2|2025-02-23 05:00:49|    Karen Wright|    shake|       Ropa|       5|         189.44|
|       3|2025-03-12 09:33:42|Elizabeth Warner|   little|Electrónica|       2|          52.83|
|       4|2025-05-04 07:43:57|     Emily Smith|     that|      Hogar|       5|         335.19|
|       5|2025-01-17 00:05:13|      David Wood|certainly|      Hogar|       1|          160.1|
+--------+-------------------+----------------+---------+-----------+--------+---------------+
only showing top 5 rows

root
 |-- id_venta: integer (nullable = true)
 |-- fecha: timestamp (nullable = true)
 |-- cliente: string (nullable = t

#  Informe de Desarrollo del Proyecto Big Data con Apache Spark

##  Objetivo General

El objetivo principal fue implementar un **pipeline completo de procesamiento Big Data** utilizando Apache Spark que integre:

- Lectura de datos estructurados y no estructurados desde múltiples fuentes.
- Procesamiento batch y en streaming.
- Consultas SQL optimizadas.
- Entrenamiento de un modelo supervisado con MLlib.
- Documentación técnica y modularización del proyecto para su potencial uso en entornos reales.

---

##  Fase 1: Análisis del caso (Lección 1)

La empresa ficticia enfrenta problemas para procesar grandes volúmenes de datos desde distintas fuentes: ventas (estructurado), sensores IoT (semi-estructurado) y logs/redes sociales (no estructurados).

###  5Vs del Big Data aplicadas
- **Volumen**: gran cantidad de registros heterogéneos.
- **Velocidad**: necesidad de procesamiento en tiempo real (streaming).
- **Variedad**: múltiples formatos (CSV, JSON, texto).
- **Veracidad**: datos ruidosos en logs o sensores.
- **Valor**: los insights impulsan decisiones comerciales.

###  Tecnologías elegidas
- **Apache Spark** (Core, SQL, Streaming, MLlib).
- Google Colab como entorno de pruebas.
- Archivos planos simulados como entrada (`ventas.csv`, `logs.txt`, `sensores.json`).

---

##  Fase 2: Diseño de arquitectura (Lección 2)

Se adoptó una arquitectura basada en **Apache Spark distribuido**:

- **Driver**: coordina la ejecución.
- **Executors**: procesan los datos en paralelo.
- **Cluster Manager**: simulado en Colab con ejecución local (modo standalone).

### Módulos utilizados:
- **Spark Core**: ejecución básica, RDDs, acciones y transformaciones.
- **Spark SQL**: consultas estructuradas.
- **Spark Streaming**: simulación de eventos en tiempo real.
- **MLlib**: entrenamiento y evaluación del modelo de predicción.

---

##  Fase 3: Procesamiento Batch con RDD y DataFrame (Lección 3)

Se trabajó con los archivos subidos:

- **RDDs**: lectura con `sc.textFile`, procesamiento de `logs.txt` con `filter`, `map`, `count`.
- **DataFrames**:
  - Lectura de `ventas.csv` con `spark.read.csv`.
  - Lectura de `sensores.json` con `spark.read.json`.
  - Limpieza y selección de columnas.
  - Conversión de tipos de datos (`cast`).
  
### Acciones utilizadas:
- `count()`, `collect()`, `show()`, `mean()`, `sortBy()`, etc.

---

##  Fase 4: Consultas SQL con Spark SQL (Lección 4)

Se registraron los DataFrames como vistas temporales y se aplicaron consultas SQL:

### Consultas ejecutadas:
- Total de ventas por categoría (`GROUP BY categoria`).
- Registros de sensores con temperatura > 32°C.
- Conteo de eventos por nivel de log (INFO, WARN, ERROR).

También se utilizaron funciones como `orderBy`, `filter`, `selectExpr`.

---

##  Fase 5: Procesamiento Streaming (Lección 5)

Para simular la entrada en tiempo real se utilizó:

- `readStream.format("socket")` en localhost:9999 (en entorno local).
- En Colab, se usó un archivo simulado que alimenta una línea por línea con `readStream`.

### Lógica aplicada:
- Conversión de texto en palabras.
- Asignación de marca de tiempo (`current_timestamp()`).
- Agregación por ventana de 10 minutos cada 5 minutos (`window`).
- Uso de watermark de 5 minutos para manejar eventos tardíos.
- Output por consola (`writeStream.format("console")`).

---

##  Fase 6: Machine Learning Escalable con MLlib (Lección 6)

Se entrenó un modelo de clasificación con Random Forest utilizando las variables `cantidad` y `precio_unitario` como features.

### Pasos ejecutados:
1. **Indexado de variables** (`StringIndexer`) para la variable objetivo.
2. **Ensamblado de features** con `VectorAssembler`.
3. **Entrenamiento del modelo** con `RandomForestClassifier`.
4. **Evaluación** con `accuracy` → el modelo alcanzó una precisión de **23.53%** (baja, pero válida como ejercicio).
5. **Guardado del modelo** con `.write().overwrite().save()` en carpeta `modelo_entrenado`.

---

##  Conclusión

Este proyecto integra de forma progresiva los conceptos de Big Data utilizando Apache Spark. Desde la lectura distribuida de archivos, pasando por consultas optimizadas, hasta procesamiento en tiempo real y entrenamiento de modelos escalables, se cubren los principales pilares del procesamiento distribuido moderno.

El enfoque modular permite escalar el pipeline y reutilizar componentes en entornos productivos reales.

---

##  Archivos generados

| Nombre                     | Descripción                                    |
|---------------------------|------------------------------------------------|
| `notebook_01_batch.ipynb` | Lectura batch con RDDs y DataFrames            |
| `notebook_02_sql.ipynb`   | Consultas SQL con Spark SQL                    |
| `notebook_03_streaming.ipynb` | Procesamiento en tiempo real (streaming)   |
| `notebook_04_ml.ipynb`    | ML escalable con MLlib                         |
| `modelo_entrenado/`       | Carpeta de modelo guardado con Random Forest   |
| `ventas.csv`              | Datos estructurados (ventas simuladas)         |
| `sensores.json`           | Datos IoT simulados                            |
| `logs.txt`                | Logs de aplicación                             |
| `README.md`               | Documentación técnica                          |

---

##  Recomendaciones futuras

- Aumentar volumen real de datos para mejorar entrenamiento del modelo.
- Integrar fuentes en tiempo real como Kafka o MQTT.
- Usar Spark en clúster real (AWS EMR, Databricks).
- Validar hiperparámetros del modelo con `CrossValidator`.

#Diagrama de Arquitectura del Pipeline Big Data

```
+---------------------------------------------------------------+
|                  FUENTES DE DATOS (Input Sources)             |
+------------------+---------------------+----------------------+
|   ventas.csv     |   sensores.json     |      logs.txt        |
| (datos estruct.) | (IoT semi-estruc.)  | (texto no estruct.)  |
+------------------+---------------------+----------------------+

                         ↓ (Lectura inicial)

+---------------------------------------------------------------+
|     INGESTA Y PROCESAMIENTO BATCH CON SPARK (RDD + SQL)       |
+------------------+----------------------+---------------------+
|   RDDs           |   DataFrames         |   Spark SQL         |
| - textFile()     | - read.csv/json()    | - spark.sql()       |
| - map(), filter()| - withColumn()       | - consultas SQL      |
| - reduce(), etc. | - groupBy(), agg()   | - UDFs               |
+------------------+----------------------+---------------------+

                         ↓ (Datos transformados)

+---------------------------------------------------------------+
|        PROCESAMIENTO EN STREAMING CON STRUCTURED STREAMING   |
+---------------------+----------------------+------------------+
| Fuente: socket/text | Watermark            | Ventanas         |
| - readStream()      | - withWatermark()    | - window()       |
| - outputMode()      | - handle eventos     | - groupBy + count|
+---------------------+----------------------+------------------+

                         ↓ (Flujo de datos procesado)

+---------------------------------------------------------------+
|   MACHINE LEARNING ESCALABLE CON MLlib (CLASIFICACIÓN)        |
+-------------------+-----------------------+-------------------+
| Preparación de DF |   Entrenamiento       |  Evaluación       |
| - VectorAssembler | - RandomForestClassifier | - accuracy()   |
| - StringIndexer   | - .fit(), .transform()  | - Evaluator      |
| - Normalización   |                       | - Guardado modelo |
+-------------------+-----------------------+-------------------+

                         ↓

+---------------------------------------------------------------+
|             SALIDA Y ALMACENAMIENTO FINAL                     |
+----------------------------+----------------------------------+
| Resultados consolidados    | Modelo guardado en disco (.save)|
| - show(), write.csv/json   | - predicciones futuras           |
+----------------------------+----------------------------------+
```
